# EHR Mamba Embedding on MIMIC-IV

This notebook trains the EHR Mamba model with the full paper §2.2 embedding on
MIMIC-IV in-hospital mortality prediction.

The pipeline uses three project files:
- `mortality_prediction_ehrmamba_mimic4.py` — `MIMIC4EHRMambaMortalityTask`, `LabQuantizer`, `collate_ehr_mamba_batch`
- `ehrmamba_embedding.py` — `EHRMambaEmbedding` (7-component §2.2 / Eq. 1 embedding)
- `ehrmamba_v2.py` — `EHRMamba` model


In [10]:
from pyhealth.datasets import MIMIC4EHRDataset, split_by_sample
from pyhealth.trainer import Trainer
from pyhealth.tasks.mortality_prediction_ehrmamba_mimic4 import (
    MIMIC4EHRMambaMortalityTask,
    LabQuantizer,
    collate_ehr_mamba_batch,
)
from pyhealth.models.ehrmamba_v2 import EHRMamba
from torch.utils.data import DataLoader
import torch


## 1. Load MIMIC-IV Dataset

Set `DEV_MODE = False` to use the full dataset.


In [11]:
DATA_ROOT = '../../data'
CACHE_ROOT = '../../data/cache'

dataset = MIMIC4EHRDataset(
    root=DATA_ROOT,
    cache_dir=CACHE_ROOT,
    tables=['patients', 'admissions', 'procedures_icd', 'prescriptions', 'labevents'],
    dev=True,
)


Using default EHR config: C:\Users\aravn\OneDrive\Desktop\EHR_DLH_PROJECT\PyHealth\pyhealth\datasets\configs\mimic4_ehr.yaml
Memory usage Before initializing mimic4_ehr: 899.8 MB
Duplicate table names in tables list. Removing duplicates.
Initializing mimic4_ehr dataset from ../../data (dev mode: True)
Using provided cache_dir: ..\..\data\cache\3f9dcf25-0255-5984-9011-4ea90b3146e4
Memory usage After initializing mimic4_ehr: 899.8 MB


## 2. Fit Lab Quantizer and Set Task

`LabQuantizer` bins continuous lab values into 5 quantile tokens per `itemid`
(paper Appendix B). `MIMIC4EHRMambaMortalityTask` builds the §2.1 token sequence:
`[CLS] [VS] events [VE] [REG] [W?] ...` with in-hospital mortality as the label.


In [12]:
# Fit LabQuantizer on all patients (fit only on train split to avoid leakage in production)
lab_quantizer = LabQuantizer(n_bins=5)
lab_quantizer.fit_from_patients(list(dataset.iter_patients()))


task = MIMIC4EHRMambaMortalityTask(
    min_age=18,
    lab_quantizer=lab_quantizer
)
sample_dataset = dataset.set_task(task)
train_dataset, val_dataset, test_dataset = split_by_sample(
    sample_dataset, ratios=[0.5, 0.2, 0.3]
)


Found cached event dataframe: ..\..\data\cache\3f9dcf25-0255-5984-9011-4ea90b3146e4\global_event_df.parquet
Setting task MIMIC4EHRMambaMortality for mimic4_ehr base dataset...
Task cache paths: task_df=..\..\data\cache\3f9dcf25-0255-5984-9011-4ea90b3146e4\tasks\MIMIC4EHRMambaMortality_b7dccb41-d2ec-5f03-a893-81d6ed770d2f\task_df.ld, samples=..\..\data\cache\3f9dcf25-0255-5984-9011-4ea90b3146e4\tasks\MIMIC4EHRMambaMortality_b7dccb41-d2ec-5f03-a893-81d6ed770d2f\samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Applying task transformations on data with 1 workers...
Detected Jupyter notebook environment, setting num_workers to 1
Single worker mode, processing sequentially
Worker 0 started processing 16 patients. (Polars threads: 12)


  0%|          | 0/16 [00:00<?, ?it/s]

Rank 0 inferred the following `['bytes']` data format.


100%|██████████| 16/16 [00:00<00:00, 73.92it/s]

Worker 0 finished processing patients.
Fitting processors on the dataset...
Label label vocab: {0: 0, 1: 1}
Processing samples and saving to ..\..\data\cache\3f9dcf25-0255-5984-9011-4ea90b3146e4\tasks\MIMIC4EHRMambaMortality_b7dccb41-d2ec-5f03-a893-81d6ed770d2f\samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld...
Applying processors on data with 1 workers...
Detected Jupyter notebook environment, setting num_workers to 1
Single worker mode, processing sequentially
Worker 0 started processing 16 samples. (0 to 16)



  0%|          | 0/16 [00:00<?, ?it/s]

Rank 0 inferred the following `['str', 'str', 'no_header_tensor:18', 'no_header_tensor:18', 'no_header_tensor:1', 'no_header_tensor:1', 'no_header_tensor:18', 'no_header_tensor:18', 'no_header_tensor:1']` data format.


100%|██████████| 16/16 [00:00<00:00, 340.70it/s]

Worker 0 finished processing samples.
Cached processed samples to ..\..\data\cache\3f9dcf25-0255-5984-9011-4ea90b3146e4\tasks\MIMIC4EHRMambaMortality_b7dccb41-d2ec-5f03-a893-81d6ed770d2f\samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld


## 3. Create Data Loaders

`collate_ehr_mamba_batch` right-pads variable-length sequences and stacks the six
EHR Mamba tensor fields (`input_ids`, `token_type_ids`, `time_stamps`, `ages`,
`visit_orders`, `visit_segments`) into `(B, L_max)` tensors.


In [13]:
train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=False,
    collate_fn=collate_ehr_mamba_batch,
)
val_loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False,
    collate_fn=collate_ehr_mamba_batch,
)
test_loader = DataLoader(
    test_dataset, batch_size=32, shuffle=False,
    collate_fn=collate_ehr_mamba_batch,
)


## 4. Initialize EHRMamba Model

`use_ehr_mamba_embedding=True` replaces PyHealth's default embedding with
`EHRMambaEmbeddingAdapter(EHRMambaEmbedding(...))` — the full 7-component
Equation 1 embedding from §2.2.


In [14]:
model = EHRMamba(
    dataset=sample_dataset,
    embedding_dim=128,
    num_layers=2,
    state_size=16,
    conv_kernel=4,
    dropout=0.1,
    use_ehr_mamba_embedding=True,
)
trainer = Trainer(model=model, metrics=['roc_auc', 'pr_auc'])
print(model)


EHRMamba(
  (embedding_model): EHRMambaEmbeddingAdapter(
    (embedding): EHRMambaEmbedding(
      (word_embeddings): Embedding(954, 128, padding_idx=0)
      (token_type_embeddings): Embedding(10, 128)
      (visit_order_embeddings): Embedding(512, 128)
      (visit_segment_embeddings): VisitEmbedding(
        (embedding): Embedding(3, 128)
      )
      (position_embeddings): Embedding(4096, 128)
      (time_embeddings): TimeEmbeddingLayer()
      (age_embeddings): TimeEmbeddingLayer()
      (scale_back_concat_layer): Linear(in_features=192, out_features=128, bias=True)
      (tanh): Tanh()
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (blocks): ModuleDict(
    (input_ids): ModuleList(
      (0-1): 2 x MambaBlock(
        (norm): RMSNorm()
        (in_proj): Linear(in_features=128, out_features=512, bias=True)
        (conv1d): Conv1d(256, 256, kernel_size=(4,), stride=(1,), padding=(3,))
        (o

## 5. Train

In [15]:
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=50,
    monitor='roc_auc',
    optimizer_params={'lr': 1e-4},
)


Training:
Batch size: 32
Optimizer: <class 'torch.optim.adam.Adam'>
Optimizer params: {'lr': 0.0001}
Weight decay: 0.0
Max grad norm: None
Val dataloader: <torch.utils.data.dataloader.DataLoader object at 0x0000026B1D361490>
Monitor: roc_auc
Monitor criterion: max
Epochs: 50
Patience: None



Epoch 0 / 50:   0%|          | 0/1 [00:00<?, ?it/s]

--- Train epoch-0, step-1 ---
loss: 0.6170


Evaluation: 100%|██████████| 1/1 [00:00<00:00,  8.80it/s]

--- Eval epoch-0, step-1 ---
roc_auc: 0.5000
pr_auc: 0.5000
loss: 0.6507
New best roc_auc score (0.5000) at epoch-0, step-1



...

--- Eval epoch-48, step-49 ---
roc_auc: 0.5000
pr_auc: 0.5000
loss: 0.9365



Epoch 49 / 50:   0%|          | 0/1 [00:00<?, ?it/s]

--- Train epoch-49, step-50 ---
loss: 0.0115


Evaluation: 100%|██████████| 1/1 [00:00<00:00, 10.03it/s]

--- Eval epoch-49, step-50 ---
roc_auc: 0.5000
pr_auc: 0.5000
loss: 0.9552
Loaded best model


## 6. Evaluate on Test Set

In [16]:
trainer.evaluate(test_loader)


Evaluation: 100%|██████████| 1/1 [00:00<00:00,  4.73it/s]
c:\Users\aravn\OneDrive\Desktop\EHR_DLH_PROJECT\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\aravn\OneDrive\Desktop\EHR_DLH_PROJECT\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


{'roc_auc': nan, 'pr_auc': 0.0, 'loss': 0.6878067851066589}

## 7. Get Patient Embeddings

Pass `embed=True` to retrieve pooled patient representations from the last
Mamba block. These can be used for downstream tasks or visualisation.


In [9]:
model.eval()
with torch.no_grad():
    batch = next(iter(test_loader))
    output = model(**batch, embed=True)
    embeddings = output['embed']
    print(f'Patient embeddings shape: {embeddings.shape}')


Patient embeddings shape: torch.Size([4, 128])
